In [1]:
from pyspark.sql import SparkSession
from pathlib import Path

spark = (
    SparkSession.builder
    .appName("TP4-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)

print("Version Spark :", spark.version)

26/11/19 16:13:56 WARN Utils: Your hostname, MacBookPro-Fouad resolves to a loopback address: 127.0.0.1; using 10.74.1.164 instead (on interface en0)
26/11/19 16:13:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 16:13:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version Spark : 3.5.6


In [2]:
chemin_gold = (
    Path.home()
    / "retail-intelligence"
    / "tp3"
    / "data"
    / "gold"
    / "kpi_multicanal"
)

print("Chemin :", chemin_gold)
print("Dossier existant :", chemin_gold.exists())

df_kpi_multicanal = spark.read.parquet(str(chemin_gold))

df_kpi_multicanal.show(truncate=False)
df_kpi_multicanal.printSchema()

print("Nombre de lignes :", df_kpi_multicanal.count())

Chemin : /Users/elkourafouad/retail-intelligence/tp3/data/gold/kpi_multicanal
Dossier existant : True
+---------------+-----------------+------------------+-------+
|quantite_totale|montant_net_total|benefice_net_total|canal  |
+---------------+-----------------+------------------+-------+
|33755474       |1150529586.37    |-552553374.82     |Magasin|
|33724378       |1535374825.67    |-168576494.45     |Web    |
+---------------+-----------------+------------------+-------+

root
 |-- quantite_totale: long (nullable = true)
 |-- montant_net_total: decimal(28,2) (nullable = true)
 |-- benefice_net_total: decimal(28,2) (nullable = true)
 |-- canal: string (nullable = true)

Nombre de lignes : 2


In [ ]:
from pyspark.sql import functions as F

df_analyse_canaux = (
    df_kpi_multicanal
    .withColumn(
        "montant_net_millions",
        F.round(F.col("montant_net_total") / 1000000, 2)
    )
    .withColumn(
        "benefice_net_millions",
        F.round(F.col("benefice_net_total") / 1000000, 2)
    )
    .orderBy(F.desc("montant_net_total"))
)

df_analyse_canaux.show(truncate=False)